# Exercise 3: Image Retrieval Using Embeddings

**CAS Deep Learning — Computer Vision**


## Learning objectives
After this exercise you should be able to:
- Extract embedding vectors from a pretrained vision model
- Use cosine similarity to find nearest neighbours in embedding space
- Build a simple image retrieval pipeline
- Compare embedding quality between a supervised and a self-supervised model

**Fill in code where you see `# YOUR CODE HERE`.**

<div style="background-color:#fff3cd;border-left:6px solid #ff9900;padding:14px 18px;margin:12px 0;border-radius:4px;color:#222">
<div style="font-size:18px;font-weight:bold">⚡ This exercise needs a GPU runtime</div>
<div style="margin-top:8px">
<b>In Google Colab:</b> click <code>Runtime → Change runtime type → T4 GPU</code> (free tier).<br>
</div>
</div>

## Setup Colab / Local Paths

The following setup sets the default data path: Make sure to check and adapt `DATA_PATH`.

If working in Colab: **Save the notebook to your personal Google Drive to persist changes**.

Packages not in the Colab environment will be installed as well.

In [ ]:
import sys
from pathlib import Path

from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"

IN_COLAB = "google.colab" in sys.modules
print(f"In Colab: {IN_COLAB}")


if IN_COLAB:
    print("Installing additional packages to Colab Environment")
    !pip install -q timm gdown huggingface_hub

    ROOT_PATH = Path("/content")
    DATA_PATH = ROOT_PATH.joinpath("data")

else:
    import pyrootutils

    ROOT_PATH = pyrootutils.setup_root(
        search_from=".",
        indicator="pyproject.toml",
        project_root_env_var=True,
        dotenv=True,
        pythonpath=True,
        cwd=True,
    )

    DATA_PATH = ROOT_PATH.joinpath("data")

print(f"Creating {DATA_PATH} if it does not exist")
DATA_PATH.mkdir(parents=True, exist_ok=True)

If you want to mount your Google Drive you can use the following function. This will enable you to persist data. However, it is not recommended to use this for model training because of its slow latency.

In [ ]:
def mount_drive(
    drive_mount_path: str = "/content/drive", data_path: str = "MyDrive/cas-dl-compvis"
) -> tuple[Path, Path]:
    from google.colab import drive

    root_path = Path(drive_mount_path)
    drive.mount(str(root_path))
    data_path = root_path.joinpath(data_path)
    return root_path, data_path


# GDRIVE_ROOT_PATH, GDRIVE_DATA_PATH = mount_drive()

## Imports

In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np
import timm
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import ImageFolder
from tqdm.notebook import tqdm
from transformers import CLIPModel, CLIPProcessor

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using: {device} | Colab: {IN_COLAB}")

if IN_COLAB and device == "cpu":
    from IPython.display import HTML, display

    display(
        HTML("""
    <div style="background:#ffebee;border:3px solid #d32f2f;border-radius:8px;
                padding:16px;margin:12px 0;font-size:16px;color:#000">
        <div style="font-size:20px;font-weight:bold;color:#d32f2f">
            ⚠️ NO GPU DETECTED — embedding extraction will be very slow
        </div>
        <div style="margin-top:8px">
            Switch to a GPU runtime: <b>Runtime → Change runtime type → T4 GPU</b>
            (free tier), then <b>Runtime → Run all</b>.
        </div>
    </div>
    """)
    )

## Functions

We define helper functions here.

In [ ]:
def ensure_dataset(
    data_path,
    archive_name,
    extracted_subdir,
    *,
    gdrive_id=None,
    hf_repo=None,
):
    """Check-then-download a dataset archive. Returns the extracted directory.

    Tries Google Drive first (gdown), falls back to Hugging Face Hub. Supports
    .tar.gz and .zip. Idempotent — skips if the extracted directory exists.
    """
    data_path = Path(data_path)
    dataset_dir = data_path / extracted_subdir
    if dataset_dir.exists():
        print(f"Dataset already present: {dataset_dir}")
        return dataset_dir

    data_path.mkdir(parents=True, exist_ok=True)
    archive_path = data_path / archive_name

    if not archive_path.exists():
        errors = []
        if gdrive_id is not None:
            try:
                import gdown

                url = f"https://drive.google.com/uc?id={gdrive_id}"
                print(f"Downloading {archive_name} from Google Drive ...")
                gdown.download(url, str(archive_path), quiet=False, fuzzy=True)
                if not archive_path.exists():
                    raise RuntimeError("gdown reported success but file is missing")
            except Exception as exc:
                errors.append(f"gdown: {exc}")
                archive_path.unlink(missing_ok=True)

        if not archive_path.exists() and hf_repo is not None:
            try:
                from huggingface_hub import hf_hub_download

                print(f"Downloading {archive_name} from Hugging Face ({hf_repo}) ...")
                cached = hf_hub_download(hf_repo, archive_name, repo_type="dataset")
                if Path(cached).resolve() != archive_path.resolve():
                    archive_path.write_bytes(Path(cached).read_bytes())
            except Exception as exc:
                errors.append(f"hf_hub_download: {exc}")

        if not archive_path.exists():
            sources = ", ".join(errors) if errors else "no sources configured"
            raise RuntimeError(f"Could not fetch {archive_name}. Tried: {sources}.")

    print(f"Extracting {archive_path.name} to {data_path} ...")
    if archive_name.endswith((".tar.gz", ".tgz")):
        import tarfile

        with tarfile.open(archive_path) as tar:
            tar.extractall(data_path)
    elif archive_name.endswith(".zip"):
        import zipfile

        with zipfile.ZipFile(archive_path) as zf:
            zf.extractall(data_path)
    else:
        raise ValueError(f"Unsupported archive format: {archive_name}")

    assert dataset_dir.exists(), (
        f"Extraction of {archive_path.name} did not produce {dataset_dir}."
    )
    print(f"Ready: {dataset_dir}")
    return dataset_dir

In [ ]:
def plot_class_grid(
    dataset: ImageFolder,
    class_names: list[str],
    n_samples: int = 3,
    title: str = "Dataset Samples",
    seed: int = 123,
    row_labels: list[str] | None = None,
):
    """Plot a grid of dataset samples — classes as rows, samples as columns.

    Args:
        row_labels: Labels shown on the left of each row. Defaults to class_names.
                    Pass an empty list or list of empty strings to suppress labels.
    """
    random.seed(seed)

    n_classes = len(class_names)
    labels = class_names if row_labels is None else row_labels
    fig, axes = plt.subplots(n_classes, n_samples, figsize=(n_samples, n_classes))

    # Ensure axes is always 2-D
    if n_classes == 1:
        axes = [axes]
    if n_samples == 1:
        axes = [[ax] for ax in axes]

    cls_indices = {idx: [] for idx in range(n_classes)}
    for i, (_, label) in enumerate(dataset.imgs):
        cls_indices[label].append(i)

    for row, (_cls_name, idx) in enumerate(
        zip(class_names, range(n_classes), strict=False)
    ):
        samples = random.sample(cls_indices[idx], min(n_samples, len(cls_indices[idx])))
        for col, sample_idx in enumerate(samples):
            image, _ = dataset[sample_idx]
            ax = axes[row][col]
            ax.imshow(image)
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_visible(False)
            if col == 0 and row < len(labels) and labels[row]:
                ax.set_ylabel(
                    labels[row], fontsize=12, rotation=0, labelpad=60, va="center"
                )

    _ = plt.suptitle(title, fontsize=14)
    _ = plt.tight_layout()
    return fig, axes


def _ax_hide(ax) -> None:
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)


def plot_retrieval_comparison(
    query_image,
    query_label: int,
    class_names: list[str],
    results: list[tuple[str, list[tuple]]],
):
    """Plot side-by-side retrieval results for multiple models.

    Args:
        query_image: The query image (PIL or array).
        query_label: Integer class index of the query.
        class_names: List of class name strings.
        results: One entry per model row — (row_label, [(image, label, score), ...]).
    """
    n_rows = len(results)
    top_k = max(len(retrieved) for _, retrieved in results)

    fig, axes = plt.subplots(n_rows, 1 + top_k, figsize=(3 * (1 + top_k), 3.5 * n_rows))

    if n_rows == 1:
        axes = [axes]

    for row, (row_label, retrieved) in enumerate(results):
        ax = axes[row][0]
        ax.imshow(query_image)
        ax.set_title(
            f"QUERY\n{class_names[query_label]}",
            fontsize=10,
            fontweight="bold",
            color="blue",
        )
        _ax_hide(ax)
        ax.set_ylabel(
            row_label,
            fontsize=12,
            fontweight="bold",
            rotation=0,
            labelpad=50,
            va="center",
        )

        for rank, (ret_image, ret_label, score) in enumerate(retrieved):
            ax = axes[row][rank + 1]
            ax.imshow(ret_image)
            ax.set_title(
                f"#{rank + 1} ({score:.2f})\n{class_names[ret_label]}",
                fontsize=9,
                color="green" if ret_label == query_label else "red",
            )
            _ax_hide(ax)

    plt.tight_layout()
    return fig, axes

## 1) Choose Your Dataset!

Here we choose the `abo_furniture` (Amazon Berkeley Objects — 6 furniture categories) dataset, organized in [ImageFolder](https://docs.pytorch.org/vision/main/generated/torchvision.datasets.ImageFolder.html) layout with train/val/test splits.


DeepFashion is available as a restricted dataset in the Follow-up section.

In [ ]:
GDRIVE_IDS = {
    "abo_furniture.tar.gz": "1ClnTzR1plXXzKQslsK4YaQzzxCHU44cK",
}

HF_REPOS = {
    "abo_furniture.tar.gz": "marco-willi/abo_furniture",
}

DS_ID = "abo_furniture"

DS_FILE = f"{DS_ID}.tar.gz"
DATASET_PATH = ensure_dataset(
    DATA_PATH,
    archive_name=DS_FILE,
    extracted_subdir="abo",  # ABO archive extracts to "abo/", not "abo_furniture/"
    gdrive_id=GDRIVE_IDS[DS_FILE],
    hf_repo=HF_REPOS.get(DS_FILE),
)
print(f"Dataset path: {DATASET_PATH}")

Let's take a look at the classes.

In [ ]:
train_raw = ImageFolder(root=DATASET_PATH / "train")
class_names = train_raw.classes

print(f"train images: {len(train_raw)}")
print(f"Categories ({len(class_names)}): {class_names}")

Now we look at samples from the different categories.

In [ ]:
fig, ax = plot_class_grid(
    dataset=train_raw,
    class_names=class_names,
    n_samples=12,
    title="ABO Furniture Samples",
    row_labels=class_names,
    seed=123,
)

**Question:** What do you observe?

<details>
<summary>Click to reveal answer</summary>

The images are quite diverse and include cut-outs, mood images, close-ups, and many other variants.

</details>

## 2) Extract Image Embeddings

Pretrained vision models can be used for more than classify images: their internal representations capture rich visual information. By extracting these representations (called **embeddings**), we can compare images based on visual similarity rather than class labels.

We start with **DINOv3**, a self-supervised model from Meta. It learned visual representations purely from unlabelled images, making its embeddings particularly good for general-purpose similarity tasks. We use the ViT-Small variant `timm/vit_small_patch16_dinov3.lvd1689m` loaded via the `timm` library.

Feel free to experiment with a different model!

In [ ]:
# Load the DINOv3 ViT-Small model via timm (classification head removed)
dino_model = timm.create_model(
    "vit_small_patch16_dinov3.lvd1689m",
    pretrained=True,
    num_classes=0,
)
dino_model = dino_model.eval().to(device)

# Resolve the preprocessing config the model was trained with
data_config = timm.data.resolve_model_data_config(dino_model)

print(f"Model: {dino_model.__class__.__name__}")
print(f"Embedding dimension: {dino_model.num_features}")
print(f"Input size: {data_config['input_size']}")
print(f"Normalization — mean: {data_config['mean']}, std: {data_config['std']}")

DINOv3 is a Vision Transformer. It splits each image into patches and processes them through transformer layers. The output includes a special **`[CLS]` token**, a single vector that summarizes the entire image. This is our embedding.

Extract the `[CLS]` token embedding for the first image in the gallery:
1. Load the image from `train_raw[0]`
2. Preprocess it using the timm transform from `data_config`
3. Pass through the model's feature extractor (use `torch.no_grad()` and `dino_model.forward_features(...)`)
4. Get the `[CLS]` token: `tokens[:, 0, :]` (index 0 is always the CLS token)
5. Store the result in a variable called `embedding`

In [ ]:
# YOUR CODE HERE
image, label = train_raw[0]

preprocess = timm.data.create_transform(**data_config, is_training=False)
inputs = preprocess(image).unsqueeze(0).to(device)

with torch.no_grad():
    tokens = dino_model.forward_features(inputs)

# The [CLS] token is the first token in the sequence
embedding = tokens[:, 0, :]

print(f"Token sequence shape: {tokens.shape}")
print(f"Embedding shape: {embedding.shape}")
print(f"Embedding dtype: {embedding.dtype}")
print(f"First 5 values: {embedding[0, :5]}")

In [ ]:
assert embedding.shape == (1, 384), f"Expected (1, 384), got {embedding.shape}"
assert embedding.dtype == torch.float32
print("OK")

**Question:** The embedding is a 384-dimensional vector. What kind of information does it encode?

<details>
<summary>Click to reveal answer</summary>

The embedding captures **high-level visual features** of the image, shape, colour, texture, spatial structure, and object identity. Unlike raw pixels (which encode low-level colour values), embeddings are **semantic**: two images of different chairs will have similar embeddings, even if their pixel values are completely different.

DINOv3 embeddings are especially good at capturing semantic similarity because the model was trained with self-supervised objectives (multi-view consistency + dense feature matching) that encourage fine-grained, view-invariant representations.

However, there will be some invariance to some transformations that can be very important for a task. So it is possible, the embeddings do not work as expected.

</details>

## 3) Build an Embedding Database

To retrieve similar images, we need embeddings for **all** images, not just one. We build an embedding database by running every image through the model and stacking the results into a matrix of shape `(N, D)`, where `N` is the number of images and `D` is the embedding dimension.

For efficiency, we process images in batches using a `DataLoader`.

In [ ]:
# Build transforms matching DINOv3's expected preprocessing.
# timm resolves input size, crop percentage, and normalization from the model.
val_transforms = timm.data.create_transform(**data_config, is_training=False)

# Transformed version of the same dataset for model input
train_dataset = ImageFolder(root=DATASET_PATH / "train", transform=val_transforms)

# Embedding extraction is fast, so we use a sizeable random subset.
# The full training split has ~20k images; 5k is enough for interesting
# retrieval while keeping runtime reasonable.
N_DATASET = min(5000, len(train_dataset))
random.seed(123)
train_indices = random.sample(range(len(train_dataset)), N_DATASET)
train_subset = Subset(train_dataset, train_indices)

train_loader = DataLoader(train_subset, batch_size=32, shuffle=False, num_workers=0)

# Each ABO product has one "primary" cut-out image (view_index=0) plus up to
# three auxiliary views (close-ups, mood images). The auxiliary views can
# look misleading in isolation, a chair detail may just show upholstery, a
# bed close-up may just show the headboard fabric. We keep all views in the
# gallery for retrieval variety, but we will pick query images from primary
# views only.
import csv

primary_image_ids = {
    row["image_id"]
    for row in csv.DictReader(Path.open(DATASET_PATH / "metadata.csv"))
    if row["is_primary"] == "True"
}
primary_positions = [
    i
    for i, raw_idx in enumerate(train_indices)
    if Path(train_dataset.imgs[raw_idx][0]).stem in primary_image_ids
]

print(f"DS size: {len(train_subset)} images")
print(f"Batches: {len(train_loader)}")
print(f"Primary (cut-out) views available as queries: {len(primary_positions)}")

Extract DINOv3 embeddings for all images. Loop over `train_loader`, pass each batch through `dino_model.forward_features(...)`, and collect the `[CLS]` token embeddings.

Store the results in:
- `dino_embeddings`: tensor of shape `(N_DATASET, 384)`
- `train_labels`: list of integer labels

Hints:
- Use `dino_model.forward_features(images)`: returns the full token sequence `(B, N_tokens, D)`
- Extract the `[CLS]` token: `tokens[:, 0, :]`
- Collect embeddings in a list, then `torch.cat()` at the end

<details>
<summary>Click to reveal solution</summary>

```python
all_embeddings = []
all_labels = []

for images, labels in tqdm(train_loader, desc="DINOv3"):
    images = images.to(device)
    with torch.no_grad():
        tokens = dino_model.forward_features(images)
    embs = tokens[:, 0, :]
    all_embeddings.append(embs.cpu())
    all_labels.extend(labels.tolist())

dino_embeddings = torch.cat(all_embeddings)
train_labels = all_labels
```

</details>

In [ ]:
# YOUR CODE HERE
all_embeddings = []
all_labels = []

for images, labels in tqdm(train_loader, desc="DINOv3"):
    images = images.to(device)
    with torch.no_grad():
        tokens = dino_model.forward_features(images)
    embs = tokens[:, 0, :]  # [CLS] token
    all_embeddings.append(embs.cpu())
    all_labels.extend(labels.tolist())

dino_embeddings = torch.cat(all_embeddings)
train_labels = all_labels

print(f"Embeddings shape: {dino_embeddings.shape}")
print(f"Labels: {len(train_labels)}")

In [ ]:
assert dino_embeddings.shape == (N_DATASET, 384), (
    f"Expected ({N_DATASET}, 384), got {dino_embeddings.shape}"
)
assert len(train_labels) == N_DATASET

# Normalize embeddings to unit length (L2 norm)
dino_embeddings = F.normalize(dino_embeddings, p=2, dim=1)

# Verify normalization
norms = dino_embeddings.norm(dim=1)
assert torch.allclose(norms, torch.ones_like(norms), atol=1e-5)
print(f"Embedding norms (should be ~1.0): min={norms.min():.4f}, max={norms.max():.4f}")
print("OK")

**Question:** Why do we typically normalize the embeddings to unit length before computing similarity?

<details>
<summary>Click to reveal answer</summary>

We normalize so that **cosine similarity reduces to a simple dot product**:

$$\text{cosine\_sim}(a, b) = \frac{a \cdot b}{\|a\| \|b\|}$$

When both vectors have unit length ($\|a\| = \|b\| = 1$), this simplifies to $a \cdot b$.

Normalization also ensures that **similarity depends on direction, not magnitude**. Without it, a longer embedding vector could appear more similar simply because of its larger magnitude, not because it represents a more similar image.


However, it is not per-se clear that normalization is better. This should be tested and depends on the model.


</details>

## 4) Nearest Neighbour Retrieval

With all images embedded, retrieval is straightforward:
1. Compute the embedding of the query image
2. Compute cosine similarity between the query and every dataset embedding
3. Return the dataset images with the highest similarity

Since our embeddings are L2-normalized, cosine similarity is just a matrix multiplication.

Implement `retrieve_similar(query_idx, embeddings, top_k=5)` that:
1. Takes the query embedding from `embeddings[query_idx]`
2. Computes cosine similarity to all other embeddings (use `@` for matrix multiply)
3. Returns the indices and similarity scores of the `top_k` most similar images (excluding the query itself)

Return a tuple `(indices, scores)` where both are 1-D tensors of length `top_k`.

<details>
<summary>Click to reveal solution</summary>

```python
def retrieve_similar(query_idx, embeddings, top_k=5):
    query = embeddings[query_idx]
    similarities = embeddings @ query
    similarities[query_idx] = -1  # exclude query itself
    scores, indices = similarities.topk(top_k)
    return indices, scores
```

</details>

In [ ]:
# YOUR CODE HERE
def retrieve_similar(query_idx, embeddings, top_k=5):
    """Retrieve the top-k most similar images to a query."""
    query = embeddings[query_idx]
    similarities = embeddings @ query  # cosine sim (embeddings are normalized)
    similarities[query_idx] = -1  # exclude the query itself
    scores, indices = similarities.topk(top_k)
    return indices, scores

In [ ]:
# Verify: retrieve top-5 for the first image
test_indices, test_scores = retrieve_similar(0, dino_embeddings, top_k=5)
assert test_indices.shape == (5,), f"Expected 5 indices, got {test_indices.shape}"
assert test_scores.shape == (5,), f"Expected 5 scores, got {test_scores.shape}"
assert 0 not in test_indices, "Query index should be excluded from results"
assert (test_scores[:-1] >= test_scores[1:]).all(), "Scores should be sorted descending"
print(f"Top-5 indices: {test_indices.tolist()}")
print(f"Top-5 scores:  {[f'{s:.3f}' for s in test_scores.tolist()]}")
print("OK")

Let's see retrieval in action. For 3 query images from different categories, we retrieve the top-5 most similar products and display them side by side.

In [ ]:
def show_retrieval(
    query_idx, embeddings, train_raw, train_indices, class_names, top_k=5
):
    """Display a query image and its top-k retrieved results."""
    indices, scores = retrieve_similar(query_idx, embeddings, top_k)

    _fig, axes = plt.subplots(1, 1 + top_k, figsize=(3 * (1 + top_k), 3.5))

    # Show query
    q_raw_idx = train_indices[query_idx]
    q_image, q_label = train_raw[q_raw_idx]
    _ = axes[0].imshow(q_image)
    _ = axes[0].set_title(
        f"QUERY\n{class_names[q_label]}", fontsize=11, fontweight="bold", color="blue"
    )
    _ = axes[0].axis("off")

    # Show retrieved images
    for rank, (ret_idx, score) in enumerate(zip(indices, scores, strict=True)):
        raw_idx = train_indices[ret_idx.item()]
        ret_image, ret_label = train_raw[raw_idx]
        ax = axes[rank + 1]
        _ = ax.imshow(ret_image)
        match = "green" if ret_label == q_label else "red"
        _ = ax.set_title(
            f"#{rank + 1} ({score:.2f})\n{class_names[ret_label]}",
            fontsize=10,
            color=match,
        )
        _ = ax.axis("off")

    _ = plt.tight_layout()
    return plt.show()

In [ ]:
# Pick 3 query images from different categories — restricted to primary
# (cut-out) views so the query is visually unambiguous.
query_indices = []
seen_labels = set()
for i in primary_positions:
    label = train_labels[i]
    if label not in seen_labels:
        query_indices.append(i)
        seen_labels.add(label)
    if len(query_indices) == 3:
        break

for q_idx in query_indices:
    fig = show_retrieval(q_idx, dino_embeddings, train_raw, train_indices, class_names)
    plt.show()

**Question:** Look at the retrieved results. Are the top matches from the same category as the query? Do they also look visually similar in terms of style, shape, or colour?

<details>
<summary>Click to reveal answer</summary>

DINOv3 typically retrieves products that are both **semantically similar** (same category: chairs retrieve chairs, lamps retrieve lamps) and **visually similar** (similar shape, colour, and style). This goes beyond simple category matching: a modern minimalist chair tends to retrieve other modern minimalist chairs, not ornate antique ones.

This is because self-supervised training encourages the model to capture fine-grained visual structure, not just category-level features. The model was never trained with furniture labels — it discovered these visual similarities on its own.

However, if the dataset is small there will be more variation and it is possible less similar items are in the top-k samples.

</details>

Let's try a few more queries, including some that might be more challenging.

In [ ]:
# Show 3 more random queries (primary views only)
random.seed(123)
extra_queries = random.sample(primary_positions, 3)

for q_idx in extra_queries:
    fig = show_retrieval(q_idx, dino_embeddings, train_raw, train_indices, class_names)
    plt.show()

## 5) Compare with a Supervised Model

DINOv3 was trained **without labels** (self-supervised). How does it compare to a model trained **with labels** (supervised)? We extract embeddings from a ResNet-50 pretrained on ImageNet classification and compare the retrieval quality.

**Key difference:**
- **ResNet-50 (supervised)**: trained to distinguish 1,000 ImageNet categories. Its features are optimized for classification, not similarity.
- **DINOv3 (self-supervised)**: trained to produce consistent representations across augmented views of the same image, with additional dense feature objectives. Its features are optimized for general visual understanding.

In [ ]:
# Load ResNet-50 in feature-extraction mode (no classification head)
resnet_model = timm.create_model("resnet50", pretrained=True, num_classes=0)
resnet_model = resnet_model.eval().to(device)

# num_classes=0 removes the classification head and returns
# the global average pooled features directly
dummy = torch.randn(1, 3, 224, 224, device=device)
with torch.no_grad():
    dummy_out = resnet_model(dummy)
print(f"ResNet-50 embedding dimension: {dummy_out.shape[1]}")

Extract ResNet-50 embeddings for the same gallery images. Loop over `train_loader`, pass each batch through `resnet_model`, and collect the output embeddings.

Store the result in `resnet_embeddings`, a tensor of shape `(N_DATASET, 2048)`.

Hint: ResNet-50 with `num_classes=0` directly returns the feature vector, no need to index into `last_hidden_state`.

In [ ]:
# YOUR CODE HERE
all_embeddings = []

for images, _labels in tqdm(train_loader, desc="ResNet-50"):
    images = images.to(device)
    with torch.no_grad():
        embs = resnet_model(images)
    all_embeddings.append(embs.cpu())

resnet_embeddings = torch.cat(all_embeddings)
print(f"ResNet embeddings shape: {resnet_embeddings.shape}")

In [ ]:
assert resnet_embeddings.shape == (N_DATASET, 2048), (
    f"Expected ({N_DATASET}, 2048), got {resnet_embeddings.shape}"
)

# Normalize ResNet embeddings
resnet_embeddings = F.normalize(resnet_embeddings, p=2, dim=1)
print("OK")

Now we compare: for the same query images.

In [ ]:
for q_idx in query_indices:
    q_raw_idx = train_indices[q_idx]
    q_image, q_label = train_raw[q_raw_idx]

    dino_idx, dino_scores = retrieve_similar(q_idx, dino_embeddings, TOP_K)
    resnet_idx, resnet_scores = retrieve_similar(q_idx, resnet_embeddings, TOP_K)

    dino_hits = [
        (
            train_raw[train_indices[i.item()]][0],
            train_raw[train_indices[i.item()]][1],
            s,
        )
        for i, s in zip(dino_idx, dino_scores, strict=False)
    ]
    resnet_hits = [
        (
            train_raw[train_indices[i.item()]][0],
            train_raw[train_indices[i.item()]][1],
            s,
        )
        for i, s in zip(resnet_idx, resnet_scores, strict=False)
    ]

    fig, axes = plot_retrieval_comparison(
        q_image,
        q_label,
        class_names,
        [("DINOv3", dino_hits), ("ResNet-50", resnet_hits)],
    )
    plt.show()

**Question:** Which model retrieves more visually similar products? Why might self-supervised pretraining help for retrieval?

<details>
<summary>Click to reveal answer</summary>

**DINOv3 typically produces better retrieval results**, the retrieved images tend to be more visually coherent (similar style, shape, and colour) rather than just from the same broad category.

Why? The two models were trained with fundamentally different objectives:

- **ResNet-50 (supervised on ImageNet)** was trained to separate 1,000 object categories. Its embeddings are optimized for *classification boundaries* — it knows that chairs are different from tables, but it does not need to distinguish between styles of chairs. Two very different-looking chairs might have similar embeddings simply because they are both "chairs."

- **DINOv3 (self-supervised)** was trained to produce *consistent representations across different views* of the same image, with additional dense feature objectives that preserve fine spatial detail. This forces the model to capture fine-grained visual structure — shape, material, colour, style — because it needs to recognize that two augmented views of the same object represent the same thing.

This makes self-supervised models particularly well-suited for **retrieval**, where fine-grained similarity matters more than categorical boundaries.

*However*: Do you agree in this case?

</details>

Let's quantify the difference. We compute how often each model's top-5 results contain images from the same category as the query.

In [ ]:
# Category-match precision@5 for both models
def precision_at_k(embeddings, labels, k=5):
    """Fraction of top-k results that share the query's category."""
    correct = 0
    total = 0
    for q_idx in range(len(labels)):
        indices, _ = retrieve_similar(q_idx, embeddings, top_k=k)
        q_label = labels[q_idx]
        matches = sum(1 for idx in indices if labels[idx.item()] == q_label)
        correct += matches
        total += k
    return correct / total


dino_p5 = precision_at_k(dino_embeddings, train_labels, k=5)
resnet_p5 = precision_at_k(resnet_embeddings, train_labels, k=5)

print("Category-match Precision@5:")
print(f"  DINOv3:    {dino_p5:.1%}")
print(f"  ResNet-50: {resnet_p5:.1%}")

## 6) Visualize the Embedding Space

Embeddings live in a high-dimensional space, for example 384 or 2048 dimensions. **t-SNE** projects them to 2D while trying to preserve local neighborhoods, so images that appear close together in the plot often have similar embeddings. This is useful for visually inspecting whether categories form clusters.

In [ ]:
from sklearn.manifold import TSNE

dino_np = dino_embeddings.detach().cpu().numpy()
resnet_np = resnet_embeddings.detach().cpu().numpy()
labels_np = np.array(train_labels)

perplexity = min(30, len(labels_np) - 1)

dino_2d = TSNE(
    n_components=2,
    perplexity=perplexity,
    metric="cosine",
    learning_rate="auto",
    init="random",
    random_state=123,
).fit_transform(dino_np)

resnet_2d = TSNE(
    n_components=2,
    perplexity=perplexity,
    metric="cosine",
    learning_rate="auto",
    init="random",
    random_state=123,
).fit_transform(resnet_np)

_fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# DINOv3 embedding space
for label_idx, name in enumerate(class_names):
    mask = np.array(train_labels) == label_idx
    if mask.sum() > 0:
        _ = axes[0].scatter(
            dino_2d[mask, 0], dino_2d[mask, 1], label=name, alpha=0.6, s=20
        )
_ = axes[0].set_title("DINOv3 Embeddings (t-SNE)", fontsize=12)
_ = axes[0].legend(fontsize=9)
_ = axes[0].set_xlabel("Dim 1")
_ = axes[0].set_ylabel("Dim 2")

# ResNet-50 embedding space
for label_idx, name in enumerate(class_names):
    mask = np.array(train_labels) == label_idx
    if mask.sum() > 0:
        _ = axes[1].scatter(
            resnet_2d[mask, 0], resnet_2d[mask, 1], label=name, alpha=0.6, s=20
        )
_ = axes[1].set_title("ResNet-50 Embeddings (t-SNE)", fontsize=12)
_ = axes[1].legend(fontsize=9)
_ = axes[1].set_xlabel("Dim 1")
_ = axes[1].set_ylabel("Dim 2")

_ = plt.suptitle("Embedding Space: Do Categories Form Clusters?", fontsize=14)
_ = plt.tight_layout()
plt.show()

**Question:** How do the cluster structures compare between DINOv3 and ResNet-50?

<details>
<summary>Click to reveal answer</summary>

Both models group images by category, but the **cluster quality** often differs:

- **DINOv3** tends to produce tighter, more separated clusters. Categories are well-grouped, and within each cluster, visually similar items (e.g., similar chair styles) appear close together.
- **ResNet-50** also groups by category, but the clusters may overlap more — especially for categories that share visual features (e.g., tables and desks, or beds and sofas).

Remember: t-SNE shows only a 2D visualization of the embedding space. The full 384/2048-dimensional space may separate categories better than the projection suggests.

</details>

## Follow-up: Compare with CLIP (Vision-Language Model)

**CLIP** (Contrastive Language-Image Pretraining) was trained by OpenAI on 400M (image, text) pairs scraped from the web. Unlike DINOv3, which only sees images, CLIP learns a **joint embedding space** where an image and its matching caption end up close together.

This gives CLIP two useful properties for retrieval:
1. Its **image encoder** produces embeddings that capture both visual and semantic content.
2. Its **text encoder** lives in the same space — so we can retrieve images by typing a text query.

We use the base ViT variant, `openai/clip-vit-base-patch32`, which outputs 512-dimensional image and text embeddings.

In [ ]:
# Load CLIP (image encoder + text encoder + processor)
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_model = clip_model.eval().to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

print(f"Model: {clip_model.__class__.__name__}")
print(f"Image embedding dim: {clip_model.config.projection_dim}")

### Extract CLIP image embeddings

CLIP uses its own image preprocessing (different normalization from ImageNet). We let `clip_processor` handle that by passing PIL images directly.

In [ ]:
# Extract CLIP image embeddings for the same dataset.
# We call the vision backbone + projection directly (equivalent to
# `get_image_features`, but robust across transformers versions).
BATCH = 32
clip_batches = []

for start in tqdm(range(0, N_DATASET, BATCH), desc="CLIP"):
    batch_idx = train_indices[start : start + BATCH]
    images = [train_raw[i][0] for i in batch_idx]  # raw PIL images
    inputs = clip_processor(images=images, return_tensors="pt").to(device)
    with torch.no_grad():
        vision_out = clip_model.vision_model(**inputs)
        feats = clip_model.visual_projection(vision_out.pooler_output)
    clip_batches.append(feats.cpu())

clip_embeddings = torch.cat(clip_batches)
clip_embeddings = F.normalize(clip_embeddings, p=2, dim=1)

print(f"CLIP embeddings shape: {clip_embeddings.shape}")

### Visually compare DINOv3 vs CLIP retrieval

For the same queries, how do the retrieval results differ?

In [ ]:
# Side-by-side comparison: DINOv3 vs CLIP
TOP_K = 5

for q_idx in query_indices:
    q_raw_idx = train_indices[q_idx]
    q_image, q_label = train_raw[q_raw_idx]

    dino_idx, dino_scores = retrieve_similar(q_idx, dino_embeddings, TOP_K)
    clip_idx, clip_scores = retrieve_similar(q_idx, clip_embeddings, TOP_K)

    dino_hits = [
        (
            train_raw[train_indices[i.item()]][0],
            train_raw[train_indices[i.item()]][1],
            s,
        )
        for i, s in zip(dino_idx, dino_scores, strict=False)
    ]
    clip_hits = [
        (
            train_raw[train_indices[i.item()]][0],
            train_raw[train_indices[i.item()]][1],
            s,
        )
        for i, s in zip(clip_idx, clip_scores, strict=False)
    ]

    fig, axes = plot_retrieval_comparison(
        q_image, q_label, class_names, [("DINOv3", dino_hits), ("CLIP", clip_hits)]
    )
    plt.show()

**Question:** Which model retrieves more visually similar products?

<details>
<summary>Click to reveal answer</summary>

**DINOv3 often produces more visually coherent retrieval results**, especially when similarity depends on shape, texture, colour, material, pose, or product style.

**CLIP**, however, can be better when the query is more semantic: for example, when we care about retrieving items that match a concept such as “formal shoes”, “summer dress”, “office chair”, or “minimalist furniture”, even if the exact visual appearance differs.

Why? The two models were trained with different objectives:

- **CLIP** was trained to align images with natural-language descriptions. Its embeddings are therefore strongly semantic: images with similar textual meaning can be close together, even if they differ in colour, shape, style, or viewpoint. This is very useful for text-image search and broad concept retrieval.

- **DINOv3** was trained in a self-supervised way to produce consistent visual representations across different views of images. Its features tend to preserve more fine-grained visual structure, such as shape, layout, texture, colour, and local appearance patterns.

This makes **DINOv3 a strong default for visual similarity search**, while **CLIP is especially useful when retrieval should follow semantic concepts or text queries**.

*However*: Do you agree in this case?

</details>

In [ ]:
# Quantitative comparison: Precision@5
clip_p5 = precision_at_k(clip_embeddings, train_labels, k=5)

print("Category-match Precision@5:")
print(f"  DINOv3:    {dino_p5:.1%}")
print(f"  ResNet-50: {resnet_p5:.1%}")
print(f"  CLIP:      {clip_p5:.1%}")

### Bonus: Text-to-image retrieval

CLIP's text encoder lives in the same 512-dim space as its image encoder. That means we can encode a natural-language query and retrieve the most similar **images** using the exact same cosine-similarity machinery.

Below we encode a few text prompts and find the top-3 gallery images for each.

In [ ]:
# Text-to-image retrieval with CLIP.
# Like above, we use the text backbone + projection directly.
text_queries = [
    "a modern wooden chair",
    "a floor lamp with a fabric shade",
    "a dark leather sofa",
    "a round dining table",
]

text_inputs = clip_processor(text=text_queries, return_tensors="pt", padding=True).to(
    device
)
with torch.no_grad():
    text_out = clip_model.text_model(**text_inputs)
    text_feats = clip_model.text_projection(text_out.pooler_output)
text_feats = F.normalize(text_feats, p=2, dim=1).cpu()

TOP_K_TEXT = 3
_fig, axes = plt.subplots(
    len(text_queries), TOP_K_TEXT, figsize=(3 * TOP_K_TEXT, 3 * len(text_queries))
)

for row, (query, tfeat) in enumerate(zip(text_queries, text_feats, strict=True)):
    sims = clip_embeddings @ tfeat
    scores, indices = sims.topk(TOP_K_TEXT)

    for col, (ret_idx, score) in enumerate(zip(indices, scores, strict=True)):
        raw_idx = train_indices[ret_idx.item()]
        ret_image, ret_label = train_raw[raw_idx]
        ax = axes[row, col]
        _ = ax.imshow(ret_image)
        _ = ax.axis("off")
        if col == 0:
            _ = ax.set_title(
                f'"{query}"\n#{col + 1} ({score:.2f}) — {class_names[ret_label]}',
                fontsize=10,
                loc="left",
            )
        else:
            _ = ax.set_title(
                f"#{col + 1} ({score:.2f}) — {class_names[ret_label]}", fontsize=10
            )

_ = plt.suptitle("CLIP Text-to-Image Retrieval", fontsize=14)
_ = plt.tight_layout()
plt.show()

## Follow-up: Try a Different Domain (Open-Ended)

The retrieval pipeline we built is domain-agnostic, it works for any images, not just furniture. A natural question is: **how well does it generalize to a different product domain?**

The **DeepFashion In-Shop** dataset contains thousands of clothing images (tops, dresses, pants, etc.) with multiple views per garment. This makes it ideal for testing retrieval: can the model find the same garment photographed from a different angle?

In [ ]:
# DeepFashion download helper — only works with instructor-provided file ID.
# DeepFashion has no public HF mirror, so this stays Google Drive only.
def load_deepfashion(data_path, gdrive_file_id=None):
    """Load the DeepFashion classroom subset (internal, restricted distribution)."""
    if gdrive_file_id is None and not (Path(data_path) / "deepfashion").exists():
        raise FileNotFoundError(
            "DeepFashion dataset not found locally.\n"
            "Ask your instructor for the Google Drive file ID and pass it as\n"
            "  load_deepfashion(data_path, gdrive_file_id='<ID>')"
        )
    return ensure_dataset(
        data_path,
        archive_name="deepfashion_classroom_v1_internal.zip",
        extracted_subdir="deepfashion",
        gdrive_id=gdrive_file_id,
    )


# Uncomment and fill in the file ID to try DeepFashion:
# DEEPFASHION_PATH = load_deepfashion(DATA_PATH, gdrive_file_id="1ImwlWDEpqK1q_KMGcjSSX557CRtSFWbN")

### Open-ended tasks

If you have access to the DeepFashion dataset, try:

1. **Load the fashion dataset** using `load_deepfashion()` and create a gallery from the val split
2. **Extract DINOv3 embeddings** for the fashion gallery (reuse the same pipeline)
3. **Query a dress image**: does the model retrieve other dresses? Are they visually similar in style?
4. **Cross-domain query**: what happens if you embed a furniture image and query it against the fashion gallery? What does the model return?

These experiments test how general DINOv3's representations really are.

## Summary

In this exercise you built an image retrieval system from scratch:

1. **Embeddings** — extracted 384-dim vectors from DINOv3 that capture the visual content of each image
2. **Similarity search** — used cosine similarity to find nearest neighbours in embedding space
3. **Retrieval pipeline** — query an image, retrieve the top-k most similar products
4. **Model comparison** — DINOv3 (self-supervised) vs. ResNet-50 (supervised) — self-supervised embeddings often capture finer-grained visual similarity
5. **Follow-ups** — compared against CLIP (vision-language), and optionally transferred the pipeline to the DeepFashion domain

**Key takeaways:**
- Pretrained models are powerful feature extractors, even without task-specific training
- Self-supervised models like DINOv3 learn representations that transfer well to similarity tasks
- Vision-language models like CLIP enable **text-to-image retrieval** out of the box
- The same pipeline works across different domains (furniture, fashion, wildlife) — embeddings are general-purpose

Next: adapting pretrained models to new tasks with k-NN, linear probes, and fine-tuning!